# Operational Risk Economic Capital Demo

**Economic Capital Simulator** – Operational Risk Module  
Ajayvir Khara | Passed FRM Part I and Part II | January 2026

This notebook demonstrates the full Operational Risk simulation pipeline:

- Loading configuration and historical data
- Fitting frequency and severity models per Unit of Measure (UoM)
- Monte Carlo simulation of annual losses
- Computation of VaR and Expected Shortfall at 99.9% confidence
- Stress testing with stochastic and expert scenarios
- Visualization of loss distributions and capital uplifts
- Generation of regulatory-style Excel report

**Expected runtime**: ~30–60 seconds on a standard laptop

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Styling
plt.style.use("seaborn-v0_8-pastel")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 7)
%matplotlib inline

# Project root detection
notebook_dir = Path.cwd().resolve()
possible_roots = [
    notebook_dir,
    notebook_dir.parent,
    notebook_dir.parent.parent,
    notebook_dir.parent.parent.parent,
]
project_root = None
for p in possible_roots:
    if (p / "econ_capital").is_dir():
        project_root = p
        break
if project_root is None:
    raise RuntimeError("Could not find project root. Set project_root manually.")

print("Project root:", project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# Project imports
from econ_capital.op_risk.config import OpRiskConfig
from econ_capital.op_risk.lda_engine import lda_run_engine
from econ_capital.op_risk.stress_tests import OpRiskStressTester
from econ_capital.op_risk.scenarios import build_scenario_set_from_data
from econ_capital.op_risk.oprisk_reporting import generate_oprisk_report

## 1. Load Configuration

In [ ]:
%%time

CONFIG_PATH = project_root / "config" / "op_config.yaml"
print(f"Loading config from: {CONFIG_PATH}")

cfg = OpRiskConfig(str(CONFIG_PATH))
cfg.validate()
config = cfg.as_dict()


# Fix relative paths to absolute paths
config["frequency"]["data_path"] = str(project_root / config["frequency"]["data_path"])
config["severity"]["data_path"] = str(project_root / config["severity"]["data_path"])

print("\nKey config parameters:")
print(
    f"- Simulations: {config.get('simulation', {}).get('default_n_paths', 100_000):,}"
)
print(f"- Frequency data: {config['frequency']['data_path']}")
print(f"- Severity data: {config['severity']['data_path']}")

## 2. Run Baseline LDA Simulation

In [ ]:
%%time

print("Running baseline LDA engine...")
loss_dist, fitted_models, baseline_metrics = lda_run_engine(config)

print("\nBaseline Metrics:")
for k, v in sorted(baseline_metrics.items()):
    print(f"- {k}: £{v:,.0f}")

print("\nFitted Models Summary:")
for uom, params in fitted_models.items():
    print(f"{uom}:")
    print(f"  Frequency λ: {params['freq_params']['lambda']:.2f}")
    print(
        f"  Severity μ: {params['sev_params']['lognormal_mu']:.2f}, σ: {params['sev_params']['lognormal_sigma']:.2f}"
    )

## 3. Visualize Baseline Loss Distribution

In [ ]:
# Filter out extremes for visualization
vis_dist = loss_dist[loss_dist < np.quantile(loss_dist, 0.999)]

plt.figure(figsize=(12, 6))
sns.histplot(vis_dist, bins=100, kde=True)
plt.axvline(
    baseline_metrics["capital_999"], color="r", linestyle="--", label="99.9% VaR"
)
plt.title("Baseline Annual Loss Distribution")
plt.xlabel("Loss Amount (£)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Generate and Run Stress Scenarios

In [ ]:
%%time

print("Generating stress scenarios...")
scenario_set = build_scenario_set_from_data(
    config["frequency"]["data_path"],
    config["severity"]["data_path"],
    n_random=10,
    seed=42,
    config_dict=config,
)

tester = OpRiskStressTester(str(CONFIG_PATH))
results = tester.run_scenario_set(scenario_set, parallel=True)

print("\nTop 5 Stress Results:")
for r in results[:5]:
    print(f"- {r.name}: Uplift {r.uplift_pct:.1%} → £{r.capital_stressed:,.0f}")

## 5. Visualize Stress Uplifts

In [ ]:
uplift_df = pd.DataFrame(
    [
        {
            "Scenario": r.name,
            "Uplift %": r.uplift_pct * 100,
            "Capital": r.capital_stressed,
        }
        for r in results[:10]
    ]
).sort_values("Uplift %", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=uplift_df, x="Scenario", y="Uplift %")
plt.title("Top 10 Scenario Capital Uplifts")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Uplift Percentage")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 6. Compute Expert Judgment Capital

In [ ]:
scenario_el = 0.0
scenario_details = []

yaml_expert_scenarios = config.get("scenarios", {})
for name, s in yaml_expert_scenarios.items():
    prob = float(s.get("probability", 0.0))
    impact = float(s.get("impact", 0.0))
    annual_el = prob * impact
    scenario_el += annual_el
    scenario_details.append(
        {
            "name": name,
            "probability": prob,
            "impact": impact,
            "annual_el": annual_el,
        }
    )

total_el = sum(d["annual_el"] for d in scenario_details)

SCENARIO_CAPITAL_MULTIPLIER = 20
scenario_capital = total_el * SCENARIO_CAPITAL_MULTIPLIER

print("Expert Judgment Scenarios:")
for d in scenario_details:
    print(
        f"- {d['name']}: {d['probability']:.1%} × £{d['impact']:,.0f} → £{d['annual_el']:,.0f}"
    )
print(f"Total Scenario Capital: £{scenario_capital:,.0f}")

## 7. Generate Regulatory-Style Report

In [ ]:
%%time

import glob
import time

# Add expert details to config for reporting
config["expert_scenario_details"] = scenario_details
config["expert_scenario_capital"] = scenario_capital
config["expert_scenario_el"] = total_el

config["baseline_metrics"] = baseline_metrics

report_path = generate_oprisk_report(
    tester=tester,
    results=results,
    config=config,
    output_dir=str(project_root / "econ_capital" / "op_risk" / "reports"),
)

time.sleep(1)  # Give filesystem time to update

report_pattern = str(
    project_root
    / "econ_capital"
    / "op_risk"
    / "reports"
    / "OpRisk_Stress_Test_Report_*.xlsx"
)
reports = glob.glob(report_pattern)

if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Opening latest report: {os.path.basename(latest_report)}")
    os.startfile(latest_report)  # Windows only
else:
    print("No report found in the reports directory.")

## Next Steps / Experiments

- Modify `op_config.yaml` to change simulation paths or data sources
- Add custom scenarios to the YAML config
- Increase number of random scenarios in `build_scenario_set_from_data`
- Compare with insurance mitigation enabled
- Run the full firm-wide simulation:  
  ```bash
  python econ_capital/run_full_ec.py
  ```

See the other notebooks:
- `demo_credit.ipynb`
- `demo_market.ipynb`